# GhostPlayer: Download and Process Kaggle Data

This notebook downloads a Kaggle Big Data Bowl release, normalizes older schemas, builds GhostPlayer sequence examples, and writes graph `.npz` artifacts under `processed/`.

The default competition is `nfl-big-data-bowl-2022` because the 2025 files may not be available. Important caveat: the 2022 competition is special-teams data, not the passing-play dataset described in `PLAN.md`. The compatibility mode below can process it mechanically, but the football meaning changes from pass-defense ghosting to special-teams movement imitation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "PLAN.md").exists():
            return candidate
    raise RuntimeError("Could not find the GhostPlayer project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DOWNLOAD_DIR = DATA_DIR / "downloads"
PROCESSED_DIR = PROJECT_ROOT / "processed"

# Change this if you get access to a pass-play Big Data Bowl release later.
COMPETITION = "nfl-big-data-bowl-2022"

# 2022 compatibility mode uses non-pass terminal events so the pipeline can produce examples.
# Set this to False if you only want strict pass-play windows from PLAN.md.
LEGACY_2022_COMPATIBILITY_MODE = True
MAX_PLAYS = None  # Use a small number like 200 for a quick smoke run.

DATA_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {DATA_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Competition: {COMPETITION}")

## Install Notebook Dependencies

This cell installs the local project plus the Kaggle CLI. It supports kernels that have `pip`, and local `uv` environments that were created without `pip`.

In [ ]:
from shutil import which


def install_notebook_dependencies() -> None:
    packages = ["-e", str(PROJECT_ROOT), "kaggle>=1.6"]
    try:
        import pip  # noqa: F401
    except ModuleNotFoundError:
        uv = which("uv")
        if uv is None:
            subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
        else:
            subprocess.check_call([uv, "pip", "install", "--python", sys.executable, "-q", *packages])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


install_notebook_dependencies()
print("Notebook dependencies installed.")

## Download and Unzip Kaggle Data

This cell skips download if `games.csv`, `plays.csv`, `players.csv`, and at least one `tracking*.csv` already exist in `data/`.

In [ ]:
from zipfile import ZipFile


def kaggle_command() -> str:
    command = which("kaggle")
    if command:
        return command

    candidate = Path(sys.executable).parent / "kaggle"
    if candidate.exists():
        return str(candidate)

    raise RuntimeError("Kaggle CLI was not found after installation.")


def has_kaggle_credentials() -> bool:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    env_credentials = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    return kaggle_json.exists() or bool(env_credentials)


required_csvs = [DATA_DIR / "games.csv", DATA_DIR / "plays.csv", DATA_DIR / "players.csv"]
tracking_csvs = list(DATA_DIR.glob("tracking*.csv"))

if all(path.exists() for path in required_csvs) and tracking_csvs:
    print("Kaggle CSVs already exist locally. Skipping download.")
else:
    if not has_kaggle_credentials():
        raise RuntimeError(
            "Missing Kaggle credentials. Add ~/.kaggle/kaggle.json or set "
            "KAGGLE_USERNAME and KAGGLE_KEY, then rerun this cell."
        )

    subprocess.check_call([
        kaggle_command(),
        "competitions",
        "download",
        "-c",
        COMPETITION,
        "-p",
        str(DOWNLOAD_DIR),
    ])

    zip_files = sorted(DOWNLOAD_DIR.glob("*.zip"))
    if not zip_files:
        raise RuntimeError(f"No zip files found in {DOWNLOAD_DIR} after download.")

    for zip_path in zip_files:
        print(f"Unzipping {zip_path.name}")
        with ZipFile(zip_path) as archive:
            archive.extractall(DATA_DIR)

print("Available CSV files:")
for path in sorted(DATA_DIR.glob("*.csv")):
    print(f"- {path.name}")

## Load Raw Tables

In [ ]:
import numpy as np
import pandas as pd

from ghostplayer.data.load import RawTables, load_raw_tables
from ghostplayer.utils.config import DataPaths, EventConfig, PreprocessingConfig


paths = DataPaths(project_root=PROJECT_ROOT)
raw = load_raw_tables(paths)

print(f"games:    {raw.games.shape}")
print(f"plays:    {raw.plays.shape}")
print(f"players:  {raw.players.shape}")
print(f"tracking: {raw.tracking.shape}")
if raw.player_play is not None:
    print(f"player_play: {raw.player_play.shape}")

## Normalize 2022 or Legacy Schema

This is the adjustment step for older releases. It fills `tracking.nflId` from `players.displayName`, normalizes team columns, derives `plays.defensiveTeam` when missing, and creates missing continuous tracking columns as zeros.

In [ ]:
from ghostplayer.data.legacy import normalize_legacy_schema


legacy_result = normalize_legacy_schema(raw)
raw = legacy_result.tables
schema_report = legacy_result.report
schema_report

## Inspect and Configure Events

Strict GhostPlayer pass mode uses `ball_snap` through pass-arrival/outcome events. The 2022 special-teams release usually does not contain those terminal pass events, so compatibility mode switches to common special-teams terminal events.

In [ ]:
PASS_TERMINAL_EVENTS = (
    "pass_arrived",
    "pass_outcome_caught",
    "pass_outcome_incomplete",
    "interception",
)

SPECIAL_TEAMS_TERMINAL_EVENTS = (
    "punt_received",
    "punt_land",
    "punt_downed",
    "punt_muffed",
    "kick_received",
    "kickoff_received",
    "fair_catch",
    "touchback",
    "out_of_bounds",
    "tackle",
    "touchdown",
    "field_goal",
    "field_goal_missed",
    "extra_point",
)

event_counts = raw.tracking["event"].dropna().astype(str).value_counts()
interesting_events = event_counts[event_counts.index.str.contains("pass|snap|interception|punt|kick|tackle|touch", case=False, regex=True)]
display(interesting_events)

available_events = set(event_counts.index)
terminal_events = tuple(event for event in PASS_TERMINAL_EVENTS if event in available_events)

if not terminal_events and LEGACY_2022_COMPATIBILITY_MODE:
    terminal_events = tuple(event for event in SPECIAL_TEAMS_TERMINAL_EVENTS if event in available_events)
    print("No pass terminal events found. Using special-teams terminal events for 2022 compatibility:")
    print(terminal_events)

if not terminal_events:
    raise RuntimeError(
        "No usable terminal events found. Inspect the event list above and add the right event names "
        "to PASS_TERMINAL_EVENTS or SPECIAL_TEAMS_TERMINAL_EVENTS."
    )

config = PreprocessingConfig(
    require_dropback=not LEGACY_2022_COMPATIBILITY_MODE,
    events=EventConfig(terminal_events=terminal_events),
)

print(f"snap events: {config.events.snap_events}")
print(f"terminal events: {config.events.terminal_events}")
print(f"require_dropback: {config.require_dropback}")

## Preprocess Plays

In [ ]:
from dataclasses import asdict

from ghostplayer.data.preprocess import preprocess_pass_plays, split_game_ids


artifacts = preprocess_pass_plays(raw.plays, raw.tracking, config)

if MAX_PLAYS is not None:
    keep_keys = artifacts.plays[["gameId", "playId"]].drop_duplicates().head(MAX_PLAYS)
    artifacts.plays = artifacts.plays.merge(keep_keys, on=["gameId", "playId"], how="inner")
    artifacts.tracking = artifacts.tracking.merge(keep_keys, on=["gameId", "playId"], how="inner")
    artifacts.play_windows = artifacts.play_windows.merge(keep_keys, on=["gameId", "playId"], how="inner")

splits = split_game_ids(artifacts.plays, config.splits)

print("Drop summary:", asdict(artifacts.drop_summary))
print(f"valid plays: {artifacts.plays[['gameId', 'playId']].drop_duplicates().shape[0]}")
print(f"tracking rows after filtering: {artifacts.tracking.shape[0]}")
print(f"train games: {len(splits.train_game_ids)}")
print(f"validation games: {len(splits.validation_game_ids)}")
print(f"test games: {len(splits.test_game_ids)}")

## Build Sequence Examples

In [ ]:
from ghostplayer.data.build_sequences import build_sequence_examples


sequences, sequence_summary, position_vocab = build_sequence_examples(
    plays=artifacts.plays,
    players=raw.players,
    tracking=artifacts.tracking,
    config=config,
)

print("Sequence summary:", asdict(sequence_summary))
print(f"position vocab size: {len(position_vocab)}")
if sequences:
    first = sequences[0]
    print(f"history_continuous shape: {first.history_continuous.shape}")
    print(f"target_positions shape: {first.target_positions.shape}")
    print(f"defender nodes: {int(first.defender_mask.sum())}")
else:
    raise RuntimeError("No sequence examples were produced. Inspect schema_report, event config, and drop summaries above.")

## Serialize Graph Datasets

In [ ]:
import json

from ghostplayer.data.build_graphs import build_graph_examples, save_graph_dataset, stack_graph_examples


def examples_for_games(examples, game_ids):
    game_id_set = set(int(game_id) for game_id in game_ids)
    return [example for example in examples if int(example.metadata.game_id) in game_id_set]


split_examples = {
    "train": examples_for_games(sequences, splits.train_game_ids),
    "validation": examples_for_games(sequences, splits.validation_game_ids),
    "test": examples_for_games(sequences, splits.test_game_ids),
}

saved_paths = {}
for split_name, examples in split_examples.items():
    if not examples:
        print(f"Skipping {split_name}: no examples")
        continue

    graph_examples = build_graph_examples(examples)
    dataset = stack_graph_examples(graph_examples)
    output_path = PROCESSED_DIR / f"{split_name}_graphs.npz"
    save_graph_dataset(dataset, output_path)
    saved_paths[split_name] = str(output_path)
    print(f"{split_name}: {len(examples)} examples -> {output_path}")

metadata = {
    "competition": COMPETITION,
    "legacy_2022_compatibility_mode": LEGACY_2022_COMPATIBILITY_MODE,
    "lookback_frames": config.lookback_frames,
    "prediction_horizon": config.prediction_horizon,
    "terminal_events": list(config.events.terminal_events),
    "schema_report": schema_report,
    "drop_summary": asdict(artifacts.drop_summary),
    "sequence_summary": asdict(sequence_summary),
    "split_game_ids": {
        "train": [int(game_id) for game_id in splits.train_game_ids],
        "validation": [int(game_id) for game_id in splits.validation_game_ids],
        "test": [int(game_id) for game_id in splits.test_game_ids],
    },
    "position_vocab": position_vocab,
    "saved_paths": saved_paths,
}

metadata_path = PROCESSED_DIR / "dataset_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True))
print(f"metadata -> {metadata_path}")

## Verify Saved Artifacts

In [ ]:
from ghostplayer.data.build_graphs import load_graph_dataset


for split_name, path_text in saved_paths.items():
    dataset = load_graph_dataset(Path(path_text))
    print(split_name)
    print(f"  edge_index:          {dataset.edge_index.shape}")
    print(f"  history_continuous:  {dataset.history_continuous.shape}")
    print(f"  target_positions:    {dataset.target_positions.shape}")
    print(f"  defender_mask:       {dataset.defender_mask.shape}")
    print(f"  metadata:            {dataset.metadata.shape}")

## Optional: Train Baseline

In [ ]:
# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "ghostplayer.training.train_baseline",
#     "--train-data",
#     str(PROCESSED_DIR / "train_graphs.npz"),
#     "--validation-data",
#     str(PROCESSED_DIR / "validation_graphs.npz"),
#     "--output-dir",
#     str(PROCESSED_DIR / "models"),
#     "--epochs",
#     "20",
# ])